In [1]:
import numpy as np

# Prahl, cm^-1/M : (HbO2, Hb) aux deux longueurs d'onde de l'oxymètre
eps = {660: (319.6, 3226.56), 940: (1214.0, 693.44)}
C_HB = 150 / 64500  # mol/L, comme dans bias_model.ipynb

def mua(lam, S):
    e_ox, e_de = eps[lam]
    return 2.303 * C_HB * (S * e_ox + (1 - S) * e_de)  # cm^-1

for S in (0.97, 0.85):
    print(S, {lam: round(mua(lam, S), 2) for lam in (660, 940)})

for PI in (0.005, 0.10):
    dd_um = PI / mua(940, 0.97) * 1e4  # cm -> µm
    print(f"PI {PI*100:.1f} % -> épaisseur pulsée {dd_um:.0f} µm")

0.97 {660: 2.18, 940: 6.42}
0.85 {660: 4.05, 940: 6.08}
PI 0.5 % -> épaisseur pulsée 8 µm
PI 10.0 % -> épaisseur pulsée 156 µm


In [3]:
D = 2.0   # mm, vaisseau_diam
L = 52.0  # mm, doigt_longueur - vaisseau_fond

def stroke_volume(dd_mm):
    dA = np.pi / 4 * ((D + dd_mm)**2 - D**2)  # mm²
    return dA * L                             # mm³ = µL

for PI in (0.005, 0.10):
    dd_mm = PI / mua(940, 0.97) * 10  # cm -> mm
    print(f"PI {PI*100:.1f} % -> ΔV = {stroke_volume(dd_mm):.1f} µL")

PI 0.5 % -> ΔV = 1.3 µL
PI 10.0 % -> ΔV = 26.4 µL


In [4]:
# diamètres intérieurs typiques (BD Plastipak), à mesurer au pied à coulisse
syringes = {"1 mL": 4.78, "3 mL": 8.66, "5 mL": 12.06}  # mm
V_max = 50.0                                            # µL, cible avec marge
V_min = stroke_volume(0.005 / mua(940, 0.97) * 10)      # µL, PI = 0,5 %

for name, d in syringes.items():
    A = np.pi / 4 * d**2  # mm²
    print(f"{name}: A = {A:.1f} mm², course max {V_max/A:.2f} mm, "
          f"course min {V_min/A*1000:.0f} µm")

1 mL: A = 17.9 mm², course max 2.79 mm, course min 71 µm
3 mL: A = 58.9 mm², course max 0.85 mm, course min 22 µm
5 mL: A = 114.2 mm², course max 0.44 mm, course min 11 µm
